# Optimized pKa Prediction with XGBoost Feature Selection and QLattice Grid Search

In [1]:
# --- 1. Imports ---
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Fragments, SaltRemover
from mordred import Calculator, descriptors
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from xgboost import XGBRegressor
import feyn
from tqdm import tqdm
import logging
from func_timeout import func_timeout, FunctionTimedOut

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

In [2]:
# --- 2. Load and Process OP2 Dataset ---
train_file = './dw_data/Opt2_acidic_tr.csv'
test_file = './dw_data/Opt2_acidic_tst.csv'
target = 'pKa'
smiles_col = 'OriginalSmiles'

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)
for df in [train_df, test_df]:
    df[target] = pd.to_numeric(df[target], errors='coerce')

In [3]:
# --- 3. Convert SMILES to Mol and Strip Salts ---
saltRemover = SaltRemover.SaltRemover(defnFilename='./Salts.txt')
for df in [train_df, test_df]:
    df['Mol'] = df[smiles_col].astype(str).apply(
        lambda s: saltRemover.StripMol(Chem.MolFromSmiles(s)))

In [4]:
# --- 4. Descriptor Functions ---
calc = Calculator(descriptors, ignore_3D=True)
logging.basicConfig(filename='desc_errors.log', level=logging.INFO)

def safe_call(func, mol, timeout=1):
    try:
        return func_timeout(timeout, func, args=(mol,))
    except:
        return np.nan

def compute_rdkit(mol):
    funcs = {name: func for name, func in Descriptors.descList}
    return {k: safe_call(f, mol) for k, f in funcs.items()}

def compute_fragments(mol):
    funcs = {name: func for name, func in Fragments.__dict__.items() if name.startswith('fr_')}
    return {k: safe_call(f, mol) for k, f in funcs.items()}

In [5]:
# --- Compute RDKit + Fragment + Mordred Descriptors (Same as previous) ---
calc = Calculator(descriptors, ignore_3D=True)

def safe_call(func, mol, timeout=1):
    try:
        return func_timeout(timeout, func, args=(mol,))
    except (FunctionTimedOut, Exception) as e:
        return np.nan

def compute_rdkit_descriptors(mol):
    descriptor_funcs = {name: func for name, func in Descriptors.descList}
    if mol is None or Chem.MolToSmiles(mol) == '':
        return None
    return {name: safe_call(func, mol, timeout=1) for name, func in descriptor_funcs.items()}

def compute_fragment_counts(mol):
    frag_funcs = {name: func for name, func in Fragments.__dict__.items() if callable(func) and name.startswith('fr_')}
    if mol is None or Chem.MolToSmiles(mol) == '':
        return None
    return {name: safe_call(func, mol, timeout=1) for name, func in frag_funcs.items()}

def compute_descriptors_for_df(df):
    mols = df['Mol'].tolist()
    rdkit_list, frag_list = [], []
    for mol in tqdm(mols, desc='Computing RDKit + Fragment descriptors'):
        rdkit_desc = compute_rdkit_descriptors(mol)
        frag_desc = compute_fragment_counts(mol)
        rdkit_list.append(rdkit_desc if rdkit_desc is not None else {})
        frag_list.append(frag_desc if frag_desc is not None else {})
    rdkit_df = pd.DataFrame(rdkit_list)
    frag_df = pd.DataFrame(frag_list)
    mordred_df = calc.pandas(mols)
    full_desc_df = pd.concat([rdkit_df, frag_df, mordred_df], axis=1)
    non_zero_std_cols = full_desc_df.std(numeric_only=True)
    full_desc_df = full_desc_df[non_zero_std_cols[non_zero_std_cols > 0].index]
    full_desc_df = full_desc_df.apply(pd.to_numeric, errors='coerce')
    full_df = pd.concat([df[[target]].reset_index(drop=True), full_desc_df.reset_index(drop=True)], axis=1)
    return full_df.dropna()

train_data = compute_descriptors_for_df(train_df)
test_data = compute_descriptors_for_df(test_df)

  3%|█▊                                                                      | 59/2321 [00:01<00:23, 94.38it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Autocorrelation.py:97: RuntimeWarning: Mean of empty slice.
  return avec - avec.mean()
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Constitutional.py:80: RuntimeWarning: invalid value encountered in double_scalars
  return S / self.mol.GetNumAtoms()


 38%|███████████████████████████▎                                            | 293/774 [00:03<00:04, 98.60it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Autocorrelation.py:97: RuntimeWarning: Mean of empty slice.
  return avec - avec.mean()
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Constitutional.py:80: RuntimeWarning: invalid value encountered in double_scalars
  return S / self.mol.GetNumAtoms()


100%|████████████████████████████████████████████████████████████████████████| 774/774 [00:09<00:00, 78.28it/s]


In [ ]:
# --- 6. XGBoost Feature Selection ---
X = train_data.drop(columns=[target])
X = X.loc[:, ~X.columns.duplicated()]
y = train_data[target]
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5]
}

xgb = XGBRegressor(objective='reg:squarederror', random_state=42)
gs = GridSearchCV(xgb, param_grid, scoring='r2', cv=5, verbose=1, n_jobs=-1)
gs.fit(X, y)

best_model = gs.best_estimator_
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)

Fitting 5 folds for each of 32 candidates, totalling 160 fits


In [ ]:
# --- 7. Select Top Features for QLattice ---
N = 150
top_features = importances.head(N).index.tolist()
train_selected = train_data[[target] + top_features]
test_selected = test_data[[target] + top_features]

In [ ]:
# --- 8. QLattice Grid Search ---
ql = feyn.QLattice()
param_grid = {
    'n_epochs': [100, 200],
    'max_complexity': [100, 150, 200],
    'criterion': ['aic', 'bic']
}

results = []
for epochs in param_grid['n_epochs']:
    for complexity in param_grid['max_complexity']:
        for crit in param_grid['criterion']:
            models = ql.auto_run(train_selected, output_name=target,
                                 n_epochs=epochs, threads=16,
                                 max_complexity=complexity,
                                 criterion=crit)
            model = models[0]
            r2 = r2_score(
                test_selected[target], model.predict(test_selected))
            results.append({
                'epochs': epochs, 'complexity': complexity, 'criterion': crit, 'r2': r2
            })

results_df = pd.DataFrame(results).sort_values(by='r2', ascending=False)
print(results_df.head())